# 07 · Conditional GAN baseline (diffusion vs GAN)

A conditional **GAN** in the *same* VAE latent space with the *same* CLIP conditioning as `diffusion_v2` — a fair generative-model comparison. Inference is identical: `G(noise, aggregated_crowd) → latent → VAE decode`.

> **Honest expectation:** from-scratch GANs are unstable (mode collapse) and at this scale will likely **underperform** the diffusion model. That is itself a valid result — the GAN is a *baseline*, its value is the comparison, not winning. Watch `loss/d` and `loss/g` in W&B for divergence.

> Runtime → **GPU (A100)** recommended (base-128, 15k images). ~25–35 min. T4 works but slower.

## 1. Clone & install

In [ ]:
!git clone --branch feature/crowd-driven-visual-generation https://github.com/nishant-kumar109/gen-ai-IISc.git
%cd gen-ai-IISc/projects/crowd-driven-visual-generation
!pip install -q datasets wandb open-clip-torch
import torch; print('cuda', torch.cuda.is_available(), '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Credentials & locate checkpoints

In [ ]:
import os, getpass
from google.colab import drive; drive.mount('/content/drive')
wandb_key = getpass.getpass('W&B API key (Enter to skip): ').strip()
if wandb_key:
    import wandb; wandb.login(key=wandb_key); print('✓ W&B')
hf_token = getpass.getpass('HF token (Write scope): ').strip()
if hf_token:
    from huggingface_hub import login; login(token=hf_token); print('✓ HF')

VAE_CKPT = '/content/drive/MyDrive/crowdgen/vae/vae.pt'
if not os.path.exists(VAE_CKPT):
    from huggingface_hub import hf_hub_download, whoami
    VAE_CKPT = hf_hub_download(f"{whoami()['name']}/crowdgen-vae", 'vae.pt')
GAN_OUT = '/content/drive/MyDrive/crowdgen/gan'; os.makedirs(GAN_OUT, exist_ok=True)
AGG = '/content/drive/MyDrive/crowdgen/diffusion_v2/aggregators.pt'   # for the 4-way eval
print('VAE:', VAE_CKPT, '\nGAN_OUT:', GAN_OUT)

## 3. Train the conditional GAN
Same data + text-conditioning as `diffusion_v2` (fair comparison). Hinge loss, projection discriminator. If `loss/g` explodes or `loss/d`→0, it's collapsing — note it (a legitimate finding).

In [ ]:
!python train_gan.py --vae {VAE_CKPT} --dataset huggan/wikiart --limit 15000 \
    --image-size 64 --batch 128 --epochs 120 --lr 2e-4 --base 128 \
    --text-cond --p-text 0.5 --label-col genre \
    --out {GAN_OUT} --sample-every 10 --wandb

## 4. Inspect GAN samples
Top = real, bottom = GAN generations from each painting's label-text. Compare quality/coherence to the diffusion samples (`02`/`04`).

In [ ]:
from IPython.display import Image; Image(f'{GAN_OUT}/samples.png')

## 5. Upload to HF

In [ ]:
import os
from huggingface_hub import HfApi, create_repo, whoami
repo_id = f"{whoami()['name']}/crowdgen-gan"
create_repo(repo_id, repo_type='model', exist_ok=True, private=True)
api = HfApi()
api.upload_file(path_or_fileobj=f'{GAN_OUT}/gan.pt', path_in_repo='gan.pt', repo_id=repo_id, repo_type='model')
if os.path.exists(f'{GAN_OUT}/samples.png'):
    api.upload_file(path_or_fileobj=f'{GAN_OUT}/samples.png', path_in_repo='samples.png', repo_id=repo_id, repo_type='model')
print('✓', f'https://huggingface.co/{repo_id}')

## 6. RQ1 on the GAN — diffusion vs GAN comparison
Same 4-way RQ1 eval, now on the GAN generator. Compare the fidelity numbers to `diffusion_v2`'s (in `FINDINGS.md`). Typically expect the GAN **below** the diffusion model — the generative-model comparison for the report.

In [ ]:
!python evaluate.py --vae {VAE_CKPT} --gan-ckpt {GAN_OUT}/gan.pt --agg-ckpt {AGG} \
    --aggregators mean,centroid,deepsets,attention \
    --repeats 16 --n 300 --out {GAN_OUT}/rq1_4way
from IPython.display import Image, display
display(Image(f'{GAN_OUT}/rq1_4way/rq1_fidelity.png')); display(Image(f'{GAN_OUT}/rq1_4way/rq1_consistency.png'))

## Next
Compare GAN vs diffusion_v2 (fidelity numbers, sample quality, training stability) — write it up in `FINDINGS.md`. This completes the generative-model comparison. Remaining: the report.